# Configuración y tu primera llamada a un LLM


### ¿Qué vamos a ver?
- Tener tu **clave de acceso** (API key) a un modelo gratuito.
- Hacer tu **primera pregunta** a un LLM desde Python.
- Entender qué son los **roles** (`system`, `user`, `assistant`).
- Cambiar la **personalidad** del modelo con una sola frase.


### Conceptos clave

| Pieza | Qué es | Analogía |
|-------|--------|----------|
| **El LLM** | El modelo de IA que responde (Llama, etc.) | El "cerebro" que está en otro ordenador muy potente |
| **El proveedor** | La empresa que nos deja usar ese cerebro (OpenRouter o Groq) | La "compañía eléctrica" que nos da acceso |
| **La API key** | Tu clave personal y secreta | La "contraseña" para entrar |
| **La librería `openai`** | El código que envía tu pregunta y recibe la respuesta | El "teléfono" para llamar al cerebro |

Lo bonito: **el mismo código funciona con los dos proveedores**. Solo cambiaremos una línea.


## 2. Consigue tu clave gratuita

Necesitas **una** de estas dos (cualquiera vale, son gratis y sin tarjeta):

### Opción A — Groq (rápido, recomendado para empezar)
1. Entra en **https://console.groq.com** y crea una cuenta (email o Google).
2. En el menú, ve a **API Keys** → **Create API Key**.
3. Copia la clave (empieza por `gsk_...`). **Guárdala bien: solo se muestra una vez.**

### Opción B — OpenRouter (acceso a muchos modelos con una sola clave)
1. Entra en **https://openrouter.ai** y crea una cuenta.
2. Ve a **Keys** → **Create Key**.
3. Copia la clave (empieza por `sk-or-...`).

## 3. Instala la librería

Vamos a usar la librería **`openai`**. Aunque se llama así, **no es solo para los modelos de OpenAI**: es un "estándar" que tanto Groq como OpenRouter entienden. Por eso con ella podemos hablar con los dos.

In [ ]:
# !pip install -q openai

## 4. Introduce tu clave de forma segura

En lugar de escribir la clave directamente en el código, usamos `getpass`: aparecerá una **casilla** donde pegarla. Así **no se queda guardada** en el notebook.

Ejecuta la celda, **pega tu clave** en la casilla que aparece y pulsa Enter.

In [ ]:
from getpass import getpass

API_KEY = getpass("Pega aquí tu clave API y pulsa Enter: ")
print("Clave recibida (longitud:", len(API_KEY), "caracteres)")

Clave recibida ✅ (longitud: 56 caracteres)


## 5. Elige tu proveedor y crea el "cliente"

El **cliente** es el objeto de Python que usaremos para hablar con el modelo. Para crearlo necesita dos cosas:
- La **dirección** del proveedor (`base_url`).
- Tu **clave** (`api_key`).

Abajo solo tienes que ajustar la variable `PROVEEDOR`:
- Si usas **Groq** → déjalo en `"groq"`.
- Si usas **OpenRouter** → cámbialo a `"openrouter"`.

In [2]:
from openai import OpenAI

PROVEEDOR = "groq"

if PROVEEDOR == "groq":
    BASE_URL = "https://api.groq.com/openai/v1"
    MODELO   = "llama-3.3-70b-versatile"    
elif PROVEEDOR == "openrouter":
    BASE_URL = "https://openrouter.ai/api/v1"
    MODELO   = "meta-llama/llama-3.3-70b-instruct:free"
else:
    raise ValueError("PROVEEDOR debe ser 'groq' o 'openrouter'")

cliente = OpenAI(api_key=API_KEY, base_url=BASE_URL)

print(f"Cliente listo | Proveedor: {PROVEEDOR}  | Modelo: {MODELO}")

Cliente listo | Proveedor: groq  | Modelo: llama-3.3-70b-versatile


## Primera llamada
Llegó el momento. Vamos a enviarle una pregunta y a leer su respuesta.

Fíjate en la estructura: enviamos una **lista de mensajes**. De momento, un solo mensaje con:
- `role`: quién habla. Aquí `"user"` = tú.
- `content`: lo que dices.


In [3]:
respuesta = cliente.chat.completions.create(
    model=MODELO,
    messages=[
        {"role": "user", 
         "content": "Hola, ¿quién eres? Responde en una frase."}
    ],
)

texto = respuesta.choices[0].message.content
print(texto)

Soy un asistente virtual capacitado por Inteligencia Artificial para ofrecer información y responder a consultas de manera precisa y amigable.


### ¿Qué acaba de pasar?

1. Tu pregunta viajó por internet hasta el servidor del proveedor.
2. Allí, el modelo (Llama) la procesó y **generó** una respuesta palabra a palabra.
3. La respuesta volvió y la imprimimos.

## 7. La anatomía de la respuesta

La variable `respuesta` no es solo texto: contiene **más información útil**. La más interesante para nosotros es cuántos **tokens** (trozos de palabra) se han usado, porque en el mundo real **eso es lo que se paga**.

Ejecuta para verlo:

In [4]:
uso = respuesta.usage

print("Tokens de tu pregunta (entrada):", uso.prompt_tokens)
print("Tokens de la respuesta (salida):", uso.completion_tokens)
print("Tokens en total:", uso.total_tokens)

Tokens de tu pregunta (entrada): 49
Tokens de la respuesta (salida): 32
Tokens en total: 81


**Idea clave:** un **token** es un trozo de palabra (más o menos ¾ de una palabra en español). Cuantos más tokens, más cuesta y más tarda. Por eso, escribir prompts claros y no pedir respuestas eternas **ahorra dinero y tiempo**. 

## 8. Los tres roles: `system`, `user`, `assistant`

Una conversación con un LLM tiene tres tipos de "voces":

| Rol | Quién es | Para qué sirve |
|-----|----------|----------------|
| `system` | Las **instrucciones** iniciales | Define **cómo** debe comportarse el modelo (su personalidad, su tarea) |
| `user` | **Tú** | Tus preguntas y peticiones |
| `assistant` | El **modelo** | Sus respuestas |

El mensaje `system` es muy poderoso: es donde le decimos al modelo **quién es y qué debe hacer**. Vamos a probarlo.

In [5]:
respuesta = cliente.chat.completions.create(
    model=MODELO,
    messages=[
        {"role": "system", "content": "Eres un pirata simpático del siglo XVII. Hablas siempre como un pirata."},
        {"role": "user",   "content": "¿Me recomiendas aprender inteligencia artificial?"}
    ],
)

print(respuesta.choices[0].message.content)

¡Arrr, escucha bien, amigo mío! La inteligencia artificial, ¿eh? ¡Es un tesoro digno de considerar! En este siglo de navegación y descubrimientos, la inteligencia artificial es como un mapa del tesoro que te puede llevar a aguas tranquilas y ricas en conocimientos.

Si estás buscando ampliar tus habilidades y conquistar nuevos horizontes, ¡la inteligencia artificial es un barco que vale la pena abordar! Puedes aprender a navegar por los mares de la programación, a construir algoritmos y modelos que te permitan predecir y tomar decisiones informadas como un pirata experimentado.

Pero, ¡cuidado, amigo! La inteligencia artificial no es un juego de niños. Requiere dedicación y práctica, como aprender a manejar un barco en alta mar. Debes estar dispuesto a estudiar y practicar para dominar las habilidades necesarias.

Si estás dispuesto a embarcarte en esta aventura, ¡te prometo que el botín será rico! La inteligencia artificial puede abrirte puertas a nuevos mundos de posibilidades y opor

¿Ves cómo ha cambiado el **tono**? No hemos cambiado la pregunta, solo las **instrucciones del sistema**.

El `system` es donde, más adelante, le diremos al modelo cosas como *"eres un asistente que ayuda a planificar viajes"* o *"responde siempre en formato JSON"*. Es la base para construir agentes.